In [ ]:
import torch
# from tqdm.notebook import tqdm
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# Used to create token ids, encode data, and decode tokens
class Processor:
    def __init__(self):
        self.encodings = {}
        self.decodings = {}

    # Read data to create token ids
    def ingest(self, data=str):
        raw_chars = list(data)
        unique_chars = sorted(list(set(raw_chars)))

        # Assign each char to a token id
        for token_id, u_char in enumerate(unique_chars):
            self.encodings[u_char]    = token_id
            self.decodings[token_id] = u_char

        # Assign vocab size
        self.vocab_size = len(unique_chars)

    # Encode characters to token ids
    def encode(self, data=str):
        raw_chars = list(data)
        tokens = [self.encodings[raw_char] for raw_char in raw_chars]
        return tokens
    
    # Decode tokens into characters
    def decode(self, data=list):
        decoded_tokens = [self.decodings[token_id] for token_id in data]
        decoded_string = "".join(decoded_tokens)
        return decoded_string

In [ ]:
# Creates instance of one model
# All hyperparameters and training are done inside this object
class Model():
    def __init__(self, vocab_size, B=32, T=64, C=128, H=128, lr=1e-3):
        # Define hyper parameters
        self.B = B   # Batch size
        self.T = T   # Sequence length or Block size
        self.C = C   # Embedding dimension
        self.H = H   # Matches C as we only are implementing one head
        self.lr = lr # Learning rate
        self.vocab_size = vocab_size

        # Define matrices
        self.embedding_matrix = torch.randn(vocab_size, C, device = device) / (self.C ** 0.5)  # Holds embedding vectors for each token (Vocab_Size x C)
        self.W_q = torch.randn(H, C, device = device) / (self.C ** 0.5)    # Holds the Query weights (What we look for given input)
        self.W_k = torch.randn(H, C, device = device) / (self.C ** 0.5)    # Holds the Key weights (What the input holds/represents or has to offer)
        self.W_v = torch.randn(H, C, device = device) / (self.C ** 0.5)    # Holds the Value weights (Content that should be passed forward)
        ### Since our Head size is the same as our Embedding dimension, we can use the embedding matrix as our lm_head matrix
        ### I chose (H x C) dimensions because torches nn.Linear stores the parameter dimensions backwards like above
        ### This allows for similar computation with transposing the weights

    # Saving weights
    def save_weights(self, filepath="model_weights.pt"):
        weights = {
            "W_q": self.W_q.cpu(),
            "W_k": self.W_k.cpu(),
            "W_v": self.W_v.cpu(),
            "embedding_matrix": self.embedding_matrix.cpu(),
            "hyperparams": {
                "vocab_size": self.vocab_size,
                "B": self.B,
                "T": self.T,
                "C": self.C,
                "H": self.H,
                "lr": self.lr,
            }
        }
        torch.save(weights, filepath)
        print(f"Weights successfully saved to {filepath}")

    # Loading saved weights
    def load_weights(self, filepath="model_weights.pt"):
        checkpoint = torch.load(filepath, map_location=device)
        
        self.W_q = checkpoint["W_q"].to(device)
        self.W_k = checkpoint["W_k"].to(device)
        self.W_v = checkpoint["W_v"].to(device)
        self.embedding_matrix = checkpoint["embedding_matrix"].to(device)

        # Restore saved hyperparams so shapes match
        hp = checkpoint["hyperparams"]
        self.B, self.T, self.C, self.H = hp["B"], hp["T"], hp["C"], hp["H"]
        
        print(f"Weights successfully loaded from {filepath}")

    # Gets next token from given text
    def generate_from_text(self, text, encode_fn):
        # Convert text to token ids
        tokens = encode_fn(text)

        # Crop length if too long
        tokens = tokens[-self.T:]

        # Get B = 1
        x_ids = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0) # (1, T)
        # Get embedding vectors
        X = self.embedding_matrix[x_ids]

        # Holding copy of T temporarily
        original_T = self.T
        self.T = x_ids.shape[1]

        # Run forward pass for generation
        next_token_id = self.forward_pass(X, generation=True)

        # Restore T
        self.T = original_T

        return next_token_id.item()

    # Generate max new tokens
    def generate(self, prompt_text, processor, max_new_tokens=30):
        curr_text = prompt_text
        
        for _ in range(max_new_tokens):
            next_id = self.generate_from_text(curr_text, processor.encode)
            next_char = processor.decode([next_id])
            curr_text += next_char
            
        return curr_text

    # Create (B, T, C) matrix of randomly selected tokens from given data
    def build_train_batch(self, data):
        indices = torch.randint(len(data) - self.T - 1, (self.B,), device = device) # (B, 1)
        x_ids = torch.stack([data[ix:ix+self.T] for ix in indices]) # (B, T)
        y = torch.stack([data[ix+1:ix+self.T+1] for ix in indices]) # (B, T)

        # Replace token ids with their embedding vectors
        X = self.embedding_matrix[x_ids]    # (B, T, C)

        return x_ids, X, y
        
    # Calculate Q and K to get our pre-softmax attention matrix (A)
    def get_affinities(self, X):
        # Get our Query and Key matrices
        self.Q = X @ self.W_q.T    # (B, T, C) @ (C, H) --> (B, T, H)
        self.K = X @ self.W_k.T    # (B, T, C) @ (C, H) --> (B, T, H)
        ### This moves from the embedding dimension to our head size dimension
        ### In our case the head size is equal to the embedding dimension, so not much change happens here

        self.A = self.Q @ self.K.transpose(-2, -1)     # (B, T, H) @ (B, H, T) --> (B, T, T)
        self.A = self.A / (self.H ** 0.5)       # Scaling to prevent crazy value growth
        ### I use .tranpose here to manually swap dimension -2 and -1, or T and H, to allow correct matrix multiplication
        

    # Given we have our affinities, A, we now turn it to a lower triangle and softmax
    # We do so by setting the upper triangle to -inf
    # This ensures the softmax excludes future tokens, preventing a token from looking into the 'future'
    def softmax_attention(self):
        seq_len = self.A.shape[-1]

        # Generate a lower triangle of ones - Then set 0's to -infinity
        tril = torch.tril(torch.ones(seq_len, seq_len, device = device))
        A_shifted = self.A - self.A.max(dim=-1, keepdim=True).values  # Shifts A by subtracting max of each row to each element (Prevents overflow cases)
        A_masked = A_shifted.masked_fill(tril == 0, float('-inf'))  # --> (B, T, T) with only lower triangles maintained

        # Exponentiate all elements
        exp_vals = torch.exp(A_masked)
        exp_row_sums = exp_vals.sum(dim=-1, keepdim=True)    # Get the sums of each row (dim=-1 or dim=2)

        # Calculate softmaxes by dividing each exponentiated value by its rows sum of exponentiates 
        self.S = exp_vals / exp_row_sums   
        ### S is still --> (B, T, T) 

    # This is where our 'learning' is retrieved. 
    # Using the softmaxed attention values, S, we take a weighted average of the 'content'
    def value_aggregation(self, X):
        # Get our Value matrix
        self.V = X @ self.W_v.T      # (B, T, C) @ (C, H) --> (B, T, H)

        # Obtain our output before undoing projection
        self.O = self.S @ self.V   # (B, T, T) @ (B, T, H) --> (B, T, H)
        
        # Bring our output back to C dim and get logits
        self.Z = self.O @ self.embedding_matrix.T     # (B, T, H) @ (C, Vocab size) --> (B, T, Vocab Size)
        ### The only reason I used embedding matrix here is because H == C
        ### When the head size does NOT equal C, I must add a lm_head matrix of dimenion (Vocab size, H)

    # Softmax for our probabilities of next token
    def logits_to_p(self):
        # Exponentiate all elements
        Z_shifted = self.Z - self.Z.max(dim=-1, keepdim=True).values # Shift for overflow case
        exp_vals = torch.exp(Z_shifted)
        exp_row_sums = exp_vals.sum(dim=-1, keepdim=True)    # Get the sums of each row (dim=-1 or dim=2)

        # Calculate softmaxes by dividing each exponentiated value by its rows sum of exponentiates 
        self.P = exp_vals / exp_row_sums     # (B, T, Vocab_size)

        # Safe gaurd to prevent nan values from being used
        self.P = torch.nan_to_num(self.P, nan=1.0 / self.vocab_size)
        ### Now we have our probabilities for the next token of each token in each batch

    # Perform one forward pass to calculate predicted output
    def forward_pass(self, X, generation=False):      # X is expected (B, T, C)
        self.get_affinities(X)      # Q @ K.T  
        self.softmax_attention()    # Softmax(A)
        self.value_aggregation(X)   # O = S @ V --> Z = O * lm_head
        self.logits_to_p()          # Softmax(Z)

        # If we would like the next token to be returned
        if generation:
            last_p = self.P[:, -1, :]   # (B, vocab size)
            next_token_ids = torch.multinomial(last_p, num_samples=1)
            return next_token_ids


    
    ##### WEIGHT UPDATING #####
    # Must be called first - GZ is defined here and used in other gradients
    def gradient_W_v(self, X, y):
        # X - (B, T, C)
        # A - (B, T, T)
        # P - (B, T, Vocab size)
        # Y - (B, T)
        # W_E or W_lm - (Vocab size, C)
        # Gradient, or G, of A = Derivative of Loss wrt. A
        # Our softmaxes are based off our logits, Z = O @ W_lm
        # We know:
        # GZ = P - Y
        # GO = GZ @ W_E     (B, T, Vocab size) @ (Vocab size, H) --> (B, T, H)
        # GV = S^T @ GO     (B, T, T) @ (B, T, H) --> (B, T, H)
        # GW_V = GV^T @ X   (B, H, T) @ (B, T, C) --> (H, C) (Averaged over B)
        # GW_V = (GO^T @ S) @ X
        #      = ( (W_E^T @ GZ^T ) @ S ) @ X
        # GW_V = ( (W_E^T @ ( P^T - Y^T ) ) @ S ) @ X

        self.P = torch.clamp(self.P, 1e-9, 1.0) # Enforce a min/max of 1e-9/1.0
        # Calculating GZ
        self.GZ = self.P.clone()    # Copy of P (will be used to subtract one hot vector)
        B_indices = torch.arange(self.B, device=device).view(-1, 1)    # (B, 1)
        T_indices = torch.arange(self.T, device=device).view(1, -1)    # (1, T)
        self.GZ[B_indices, T_indices, y] -= 1                # (B, T, Vocab Size)

        # Calculating GO
        self.GO = self.GZ @ self.embedding_matrix   # (B, T, H)

        # Calculating GV
        self.GV = self.S.transpose(-2, -1) @ self.GO    # (B, T, H)

        # Calculating GW_V
        self.GW_V = self.GV.transpose(-2, -1) @ X
        self.GW_V = self.GW_V.sum(dim=0)           # Sum over B (H, C)
        return self.GW_V

        ###
        ### OLD IMPLEMENTATION * INCORRECT DERIVATION *
        ###
        # # GO = (P - Y) @ W_lm                          - GO --> (B, T, Vocab size) @ (Vocab size, C) --> (B, T, C)
        # # O = A @ V so linear derivative property says - GV = A.T @ GO   --> (B, T, T) @ (B, T, C) --> (B, T, C)
        # # V = X @ W_V                                  - GW_V = X.T @ GV --> (B, C, T) @ (B, T, C) --> (B, C, C)
        # # So GW_V = X.T ( A.T @ [( P - Y) @ W_lm])     - (B, C, T) @ [ (B, T, T) @ [ (B, T, Vocab Size) @ (Vocab Size, C) ] ]
        # #                                              - (B, C, T) @ [ (B, T, T) @ [ (B, T, C)]]
        # #                                              - (B, C, T) @ [ (B, T, C)]
        # #                                              - (B, C, C)      * Consistent *
        # self.P = torch.clamp(self.P, 1e-9, 1.0)
        # self.GZ = self.P.clone()
        # B_indices = torch.arange(self.B, device=device).view(-1, 1)    # (B, 1)
        # T_indices = torch.arange(self.T, device=device).view(1, -1)    # (1, T)
        # self.GZ[B_indices, T_indices, y] -= 1                # (B, T, Vocab Size)

        # GO = self.GZ @ self.embedding_matrix     # (B, T, C)
        # self.GV = self.A.transpose(-2, -1) @ GO  # (B, T, C)
        # GW_V = X.transpose(-2, -1) @ self.GV     # (B, C, C)

        # # We need sum of gradients across batches
        # GW_V = GW_V.sum(dim=0)              # (C, H)
        # return GW_V.T                       # (H, C) for updating W_V which is also (H, C)

    def gradient_W_qk(self, X, y):
        # X                                 - (B, T, C)
        # GZ = P - Y                        - (B, T, Vocab size)
        # W_lm or embedding matrix          - (Vocab size, C)
        # GO = GZ @ W_lm                    - (B, T, C) - Only 'C' because C equals H

        ### K and Q                         - (B, T, H)
        # W_k and W_q                       - (H, C)
        # Q = X @ W^T_q                     - (B, T, H)
        # K = X @ W^T_k                     - (B, T, H)
        # GS = GO @ V^T                     - (B, T, T)
        # GA = S * (GS - S dot GS)          - (B, T, T)
        # GQ = 1/root(H) * GA @ K           - (B, T, H)
        # GK = 1/root(H) * GA^T @ Q         - (B, T, H)
        # GW_Q = GQ^T @ X                   - (H, C)
        # GW_K = GK^T @ X                   - (H, C)
        
        # Calculating GS
        self.GS = self.GO @ self.V.transpose(-2, -1)

        # Calculating GA
        S_GS = self.S * self.GS
        rowsum = S_GS.sum(dim=-1, keepdim=True) # (B, T, 1)
        self.GA = self.S * (self.GS - rowsum)   # (B, T, T)

        # Calculating GQ and GK
        # print(f"GQ = 1/root(h) * GA @ X:\t ({self.GA.shape}^T @ {self.K.shape})\n")
        self.GQ = (1 / torch.sqrt(torch.tensor(self.H, dtype=torch.float32))) * (self.GA @ self.K)
        self.GK = (1 / torch.sqrt(torch.tensor(self.H, dtype=torch.float32))) * (self.GA.transpose(-2, -1) @ self.Q)

        # Calculating W_Q and W_K       
        self.GW_Q = self.GQ.transpose(-2, -1) @ X
        self.GW_K = self.GK.transpose(-2, -1) @ X

        # Sum out over B
        self.GW_Q = self.GW_Q.sum(dim=0)           # (H, C)
        self.GW_K = self.GW_K.sum(dim=0)           # (H, C)

        return self.GW_Q, self.GW_K

        ###
        ### OLD IMPLEMENTATION * INCORRECT DERIVATION *
        ###
        # GO = self.GZ @ self.embedding_matrix    # (Vocab size, C)
        # GA = GO @ self.V.transpose(-2, -1)      # (B, T, T)
        # A_GA = self.A * GA                      # (B, T, T) - Element wise multiplication
        # rowsums = A_GA.sum(dim=-1, keepdim=True)# (B, T, 1)
        # GS = self.A * (GA - rowsums)            # (B, T, T)
        # GS = GS / (self.H ** 0.5)            # Scaling to prevent crazy value growth

        # # Derive gradients for Q and K weights
        # self.GQ = GS @ self.K                        # (B, T, H)
        # self.GK = GS.transpose(-2, -1) @ self.Q      # (B, T, H)

        # GW_Q = self.GQ.transpose(-2, -1) @ X         # (B, H, C) or (B, C, C)
        # GW_K = self.GK.transpose(-2, -1) @ X         # (B, H, C) or (B, C, C)
        
        # GW_Q = GW_Q.sum(dim=0)                  # (H, C)
        # GW_K = GW_K.sum(dim=0)                  # (H, C)

        # return GW_Q, GW_K
    
    def gradient_W_e(self, x_ids, X, y):
        # O                 - (B, T, C)
        # Z = O @ W^T_e     - (B, T, Vocab size)
        # GZ = P - Y        - (B, T, Vocab Size)
        ## Output gradient ##
        # GW_e = (GZ)^T @ O - (B, Vocab size, C)
        ## Input gradient ##
        # GX = GQ @ W_Q + GK @ W_K + GV @ W_V   - (B, T, C)
        
        # Output gradient
        GZ_flat = self.GZ.view(-1, self.vocab_size) # Makes (B, T, Vocab size) --> (B * T, Vocab size)
        O_flat = self.O.view(-1, self.C)            # Makes (B, T, C) --> (B * T, C)

        GW_E_output = GZ_flat.T @ O_flat    # (Vocab size, B*T) @ (B*T, C) --> (Vocab size, C)

        # Input gradient
        GX = (self.GQ @ self.W_q) + (self.GK @ self.W_k) + (self.GV @ self.W_v) # (B, T, C)
        GW_E_input = torch.zeros_like(self.embedding_matrix)    # (Vocab size, C)
        GW_E_input.index_add_(0, x_ids.to(device).view(-1), GX.view(-1, self.C))

        return GW_E_output + GW_E_input

    def back_pass(self, x_ids, X, y):
        # Get gradients
        GW_V = self.gradient_W_v(X, y)
        GW_Q, GW_K = self.gradient_W_qk(X, y)
        GW_E = self.gradient_W_e(x_ids, X, y)

        # Update weights
        BT = self.B * self.T        # Keeps the gradients from exploding
        self.W_v -= self.lr * (GW_V / self.B)
        # print(f"W_q: ({self.W_q.shape})\nlr: ({self.lr})\nGW_Q: ({GW_Q.shape})\nBT: ({BT})")
        self.W_q -= self.lr * (GW_Q / self.B)
        self.W_k -= self.lr * (GW_K / self.B) 
        self.embedding_matrix -= self.lr * (GW_E / self.B)

    ##### TRAINING LOOP #####
    def train(self, data, max_iter=10000):  # Default Iterations = 10k
        for i in tqdm(range(max_iter), desc="LLM Training", total=max_iter):
            # Get random batches
            x_ids, X, y = self.build_train_batch(data)

            self.forward_pass(X)
            self.back_pass(x_ids, X, y)

            if i % 100 == 0:
                # Manual Cross-Entropy Loss: -log(probability of the correct token)
                B_idx = torch.arange(self.B).view(-1, 1)
                T_idx = torch.arange(self.T)
                correct_probs = self.P[B_idx, T_idx, y]
                loss = -torch.log(correct_probs + 1e-9).mean() # 1e-9 prevents log(0)
                print(f"Iter {i}: Loss {loss.item():.4f}")
                #print(f"\n{self.W_q, self.W_k, self.W_v, self.embedding_matrix}\n")
                
        return self.W_q, self.W_k, self.W_v, self.embedding_matrix




In [ ]:
processor = Processor()

# Read and ingest data
with open("tiny-shakespeare.txt", 'r') as f:
  data = f.read()

data = "Hello my name is Kylan. What is your father doing out here in the cold?"
processor.ingest(data)
tokenized_data_list = processor.encode(data)

# Convert into tensor
tokenized_data = torch.tensor(tokenized_data_list, dtype=torch.long)

# Get vocab size
vocab_size = processor.vocab_size

In [ ]:
model = Model(vocab_size, B=16, T=16, C=64, H=64, lr=0.05)
model.train(tokenized_data, 2000)
model.save_weights()


LLM Training:   1%|          | 13/2000 [00:00<00:15, 129.10it/s]

Iter 0: Loss 3.1469


LLM Training:   7%|▋         | 132/2000 [00:00<00:10, 172.87it/s]

Iter 100: Loss nan


LLM Training:  11%|█         | 212/2000 [00:01<00:07, 228.51it/s]

...

Iter 1800: Loss nan


LLM Training:  96%|█████████▌| 1920/2000 [00:08<00:00, 301.53it/s]

Iter 1900: Loss nan


LLM Training: 100%|██████████| 2000/2000 [00:08<00:00, 232.83it/s]

Weights successfully saved to model_weights.pt


In [ ]:
new_model = Model(vocab_size, lr=1e-3)

# Load the saved parameters
new_model.load_weights()

text = "Hello "
generated_text = new_model.generate(text, processor, max_new_tokens=40)
print(f"Generated output:\n{generated_text}")

AcceleratorError: CUDA error: unknown error
Search for `cudaErrorUnknown' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
